# 1 Descargar Ventas y Visitas (aje-prd-analytics-artifacts-s3)

In [1]:
# import psutil
# import time
# import threading
# import matplotlib.pyplot as plt

# # Variables globales
# mem_usage_gb = []
# mem_usage_percent = []
# cpu_usage_gb = []
# cpu_usage_percent = []
# timestamps = []
# running = True  # Variable de control

# # Función de monitoreo en segundo plano
# def monitor_resources():
#     global running
#     start_time = time.time()
    
#     while running:
#         # Obtener información del sistema
#         mem_info = psutil.virtual_memory()
#         cpu_percent = psutil.cpu_percent(interval=0.5)  # Medir CPU sin bloquear
#         cpu_used_gb = (cpu_percent / 100) * (psutil.virtual_memory().total / (1024**3))  # Estimación del uso de CPU en GB
        
#         # Guardar datos
#         timestamp = time.time() - start_time
#         mem_usage_gb.append(mem_info.used / (1024 ** 3))  # RAM usada en GB
#         mem_usage_percent.append(mem_info.percent)  # RAM en %
#         cpu_usage_gb.append(cpu_used_gb)  # CPU en GB
#         cpu_usage_percent.append(cpu_percent)  # CPU en %
#         timestamps.append(timestamp)
        
#         # Imprimir valores en tiempo real
#         # print(f"Tiempo: {timestamp:.2f}s | RAM: {mem_info.percent:.2f}% ({mem_usage_gb[-1]:.2f} GB) | CPU: {cpu_percent:.2f}% ({cpu_used_gb:.2f} GB)")
        
#         time.sleep(1)  # Medir cada 1 segundo

# # Iniciar monitoreo en segundo plano
# monitor_thread = threading.Thread(target=monitor_resources, daemon=True)
# monitor_thread.start()

# print("Monitoreo de CPU y RAM iniciado en segundo plano...")

In [2]:
!pip install --upgrade pip setuptools wheel
!pip install --upgrade pyarrow==17.0.0
!pip install "awswrangler[redshift]" --no-build-isolation
!pip install psycopg2-binary

In [3]:
import pandas as pd
import redshift_connector
import awswrangler as wr
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import os
import boto3
import io
import pytz
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import re
import gc

In [4]:
def comprobar_inputs():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"
    
    hoy = datetime.now(pytz.timezone("America/Lima")).date()
    errores = []

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" not in objetos:
        print("ERROR: No se encontraron archivos en la ruta especificada.")
        return

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        key = objeto["Key"]
        if key == "pedido_sugerido/data-v1/cam/ventas_sellout_cam000":
            print(f"OJO: El key {key} aplica a la validacion.")
            continue
        print(key)
        # Omitir "carpetas" en S3
        if key.endswith("/"):
            continue

        last_modified = objeto["LastModified"].date()
        size_kb = objeto["Size"] / 1024  # Convertir a KB

        # Verificar si el objeto tiene contenido real (FULL_OBJECT)
        if objeto["Size"] == 0:
            errores.append(f"ERROR: El archivo {key} está vacío.")

        # Verificar si el archivo ha sido modificado hoy
        if last_modified != hoy:
            errores.append(f"ERROR: El archivo {key} no ha sido modificado hoy ({hoy}), su ultima fecha de modificacion fue {last_modified}.")

        # Verificar si el tamaño del archivo es menor a 1 KB
        if size_kb < 1:
            errores.append(f"ALERTA: El archivo {key} tiene un tamaño menor a 1 KB ({size_kb:.2f} KB).")

    # Mostrar los errores y lanzar una excepción si es necesario
    if errores:
        for error in errores:
            print(error)
        raise ValueError("Se encontraron problemas con los archivos en S3.")
    else:
        print("Todo bien :D")

In [5]:
comprobar_inputs()

pedido_sugerido/data-v1/cam/
pedido_sugerido/data-v1/cam/D_stock_cr.csv
pedido_sugerido/data-v1/cam/D_stock_gt.csv
pedido_sugerido/data-v1/cam/D_stock_ni.csv
pedido_sugerido/data-v1/cam/D_stock_pa.csv
pedido_sugerido/data-v1/cam/maestro_productos_cam000
pedido_sugerido/data-v1/cam/ventas_cam000
pedido_sugerido/data-v1/cam/ventas_cam001
OJO: El key pedido_sugerido/data-v1/cam/ventas_sellout_cam000 aplica a la validacion.
pedido_sugerido/data-v1/cam/visitas_cam000
Todo bien :D


In [6]:
# Obtener cod SKUs y descripcion de CAM
query=f'''select * from
    comercial_cam.dim_producto
    where estado='A' and cod_compania = '0076';
    '''

con = wr.data_api.redshift.connect(cluster_id = 'dwh-cloud-storage-salesforce-prod', database = 'dwh_prod',db_user='dwhuser')
maestro_prod = wr.data_api.rds.read_sql_query( query, con)

In [7]:
maestro_prod = maestro_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
maestro_prod = maestro_prod.groupby(["cod_articulo_magic"]).first().reset_index()
maestro_prod.to_csv("Input/D_pan_masterProd.csv", index=False)

In [8]:
# maestro_prod.columns

In [9]:
# maestro_prod[~maestro_prod.desc_unidad_negocio.isnull()]

In [10]:
# maestro_prod[["cod_articulo_magic","desc_articulo"]].drop_duplicates().reset_index(drop=True)

In [11]:
# rutas_test = [1007, 1005, 1008, 1013]
# rutas_test = [1001, 1002, 1003, 1004, 1005,
#               1006, 1007, 1008, 1009, 1010,
#               1012, 1013, 1014, 1016, 16101,
#               28001, 29000, 29001]
rutas_test = [
# Piloto    
1202,1206,1361,1370,1195,1192,1146,1148,1402,1406,1311,1301,1120,1106,
# Nuevas Rutas 2025-07-21
1101,1102,1103,1104,1105,1106,1107,1108,1109,1110,1111,
1112,1113,1114,1115,1116,1117,1118,1119,1120,1160,
1161,1162,1163,1164,1165,1166,1167,1168,1169,1170,1171,
1172,1173,1174,1175,1176,1177,1178,1179,1180,1181,1182,
1183,1184,1185,1186,1187,1188,1189,1190,1191,1192,1193,
1194,1195,1196,1197,1198,1200,1201,1202,1203,1204,
1205,1215,1207,1208,1209,1210,1211,1212,1216,1206,1213,
1214,1217,1218,1219,1220,1300,1301,1302,1303,1304,
1305,1306,1307,1308,1309,1310,1311,1312,1313,1350,
1351,1352,1353,1354,1355,1356,1357,1358,1359,1360,1361,
1362,1363,1364,1365,1366,1367,1368,1369,1370,1371,1372,
1373,1374,1375,1376,1377,1378,1379,
# Quitar rutas 2025-07-21 6pm
# 3101 3131 3160 3200 3210 3300 3400
# Añadir mas rutas
1144,1145,1146,1147,1148,1149,1150,1137,1138,
1139,1140,1141,1142,1143,1131,1132,1133,1134,
1135,1136,1400,1401,1402,1403,1404,1405,
1406,1407,1408,1409,1410,1411,1412,1413,1414,
1415,1416,1417,1418,1419,1420,1421,1422,1423,
1424,1425,1426,1427,1428,1429,1430,1431,1432,
1433,1434,1435]

In [12]:
len(rutas_test)

194

In [13]:
rutas_test = list(set(rutas_test)) # quitamos rutas duplicadas
len(rutas_test)

180

In [14]:
clientes_ruta_test = []

In [15]:
def descargar_visitas():
    # Guardar clientes de las rutas
    global clientes_ruta_test

    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0 and objeto["Key"].split("/")[-1] == "visitas_cam000":
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "visitas_cam000":
                print(objeto)
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df.pais__c == "GT")
                    & (df["cod_ruta"].isin(rutas_test))
                    & (df.codigo_canal__c == 2)
                ].reset_index(drop=True)
                
                # Guardamos estos clientes para descargarlos en ventas (por si cambiaron de ruta)
                clientes_ruta_test = df["codigo_cliente__c"].unique()
                nombre_csv = f"Input/{nombre_archivo}.parquet"
                df["personalizacion"] = df["personalizacion"].astype(str)
                df.to_parquet(nombre_csv, index=False)
            else:
                continue

In [16]:
%%time
descargar_visitas()

{'Key': 'pedido_sugerido/data-v1/cam/visitas_cam000', 'LastModified': datetime.datetime(2026, 3, 30, 15, 2, 28, tzinfo=tzlocal()), 'ETag': '"957996d65c2604d1523a716d29797013-1"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 43502592, 'StorageClass': 'STANDARD'}
CPU times: user 927 ms, sys: 182 ms, total: 1.11 s
Wall time: 1.5 s


In [17]:
def descargar_ventas():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/cam/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Creamos DF que juntará los 2 archivos
    final_csv = pd.DataFrame()

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0:
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "ventas_cam000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)
            elif nombre_archivo == "ventas_cam001":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)

            else:
                continue

    nombre_csv = f"Input/ventas_cam_all.parquet"
    final_csv.to_parquet(nombre_csv, index=False)

In [18]:
%%time
descargar_ventas()

{'ResponseMetadata': {'RequestId': 'AWBJGKM38C4J3R1Z', 'HostId': 'eJ2DxiL0poHltXzhBEw2iJcINJhiaQVvok9f/A6tlsbnLYJQoboBipnf3jGHPIQ9RDddndE62qjkBTt9+KiiJFXXEyiAsmyD', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'eJ2DxiL0poHltXzhBEw2iJcINJhiaQVvok9f/A6tlsbnLYJQoboBipnf3jGHPIQ9RDddndE62qjkBTt9+KiiJFXXEyiAsmyD', 'x-amz-request-id': 'AWBJGKM38C4J3R1Z', 'date': 'Mon, 30 Mar 2026 21:30:54 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/cam/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 33, 15, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/cam/D_stock_cr.csv', 'LastModified': datetime.datetime(2026, 3, 30, 14, 3, 37, tzinfo=tzlocal()), 'ETag': '"7dda2ed19935555f934134f38e064ffd"', 'ChecksumAlgorithm': ['CRC32'], 'Checksu

/tmp/ipykernel_21325/2670914792.py:28: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(content), sep=";")


CPU times: user 2min 14s, sys: 26 s, total: 2min 40s
Wall time: 4min 34s


## 1.1 Juntar VENTAS y VISITAS

In [19]:
import pandas as pd
from datetime import datetime, timedelta

In [20]:
pan_ventas = pd.read_parquet("Input/ventas_cam_all.parquet")
pan_visitas = pd.read_parquet("Input/visitas_cam000.parquet")

In [21]:
pan_ventas = pan_ventas[pan_ventas.cod_pais=="GT"].reset_index(drop=True)

In [22]:
pan_visitas = pan_visitas[(pan_visitas.codigo_canal__c==2)&(pan_visitas.compania__c=="0076")].reset_index(drop=True)

In [23]:
pan_ventas["cod_articulo_magic"] = pan_ventas["id_producto"].str.split("|").str[-1]
pan_ventas["cod_articulo_magic"] = pan_ventas["cod_articulo_magic"].astype(int)

In [24]:
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print(hoy)
# Filtrar solo las filas con la fecha de hoy
pan_ventas = pan_ventas[pan_ventas['fecha_proceso'] == hoy]

20260330


In [25]:
# Obtener la fecha y año actual
current_date = datetime.now(pytz.timezone("America/Lima"))
# Formatear la fecha en el formato "YYYYMM"
formatted_date = current_date.strftime("%Y-%m-%d")
formatted_date

'2026-03-30'

In [26]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Guatemala/Input/visitas_cam000_{formatted_date}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(pan_visitas, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Guatemala/Input/visitas_cam000_2026-03-30.csv'],
 'partitions_values': {}}

In [27]:
PS_VAL_total_skus_venta = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_venta

119

In [28]:
# PAN VENTAS
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["id_cliente"] = (
    "CAM"
    + "|"
    + pan_ventas["cod_compania"].astype(str)
    + "|"
    + pan_ventas["cod_cliente"].astype(str)
)

# PAN VISITAS
pan_visitas["compania__c"] = (
    pan_visitas["compania__c"].astype(str).apply(lambda x: str((x)).rjust(4, "0"))
)
pan_visitas["id_cliente"] = (
    "CAM"
    + "|"
    + pan_visitas["compania__c"].astype(str)
    + "|"
    + pan_visitas["codigo_cliente__c"].astype(str)
)

In [29]:
PS_VAL_ultima_venta = pan_ventas["fecha_liquidacion"].sort_values().unique()[-1:]
PS_VAL_ultima_venta

array(['2026-03-28'], dtype=object)

In [30]:
PS_VAL_ultima_visita = pan_visitas["ultima_visita"].sort_values().unique()[-2:-1]
PS_VAL_ultima_visita

array(['2026-03-28'], dtype=object)

In [31]:
PS_VAL_total_clientes_venta = pan_ventas["id_cliente"].nunique()
PS_VAL_total_clientes_venta

50659

In [32]:
PS_VAL_total_clientes_visita = pan_visitas["id_cliente"].nunique()
PS_VAL_total_clientes_visita

50858

In [33]:
pan_visitas = pan_visitas[pan_visitas.codigo_canal__c == 2].reset_index(drop=True)

In [34]:
PS_VAL_total_clientes_visita_trad = pan_visitas["id_cliente"].nunique()
PS_VAL_total_clientes_visita_trad

50858

In [35]:
# Eliminar Duplicados y quedarse con la ultima visita más reciente
visita_default_semana_atras = (datetime.now(pytz.timezone("America/Lima")) - timedelta(days=7)).strftime("%Y-%m-%d")
pan_visitas["ultima_visita"] = pan_visitas["ultima_visita"].fillna(
    visita_default_semana_atras
)
pan_visitas = (
    pan_visitas.sort_values(["id_cliente", "ultima_visita"], ascending=False)
    .groupby("id_cliente")
    .head(1)
)

In [36]:
# Quitamos cod_modulo de ventas para luego quedarnos con el de visitas
# pan_ventas = pan_ventas.drop(["cod_modulo","cod_ruta"], axis=1)

In [37]:
# pan_ventas=pd.merge(pan_ventas,pan_visitas[["id_cliente","dias_de_visita__c","periodo_de_visita__c","ultima_visita","cod_ruta","cod_modulo"]],on=["id_cliente"],how="outer").drop_duplicates()

In [38]:
# Combina ventas y visitas (nos quedamos con ruta y módulo de visitas si es nulo, usamos el valor de ventas)
df_merged = pd.merge(
    pan_ventas,
    pan_visitas[
        [
            "id_cliente",
            "dias_de_visita__c",
            "periodo_de_visita__c",
            "ultima_visita",
            "cod_ruta",
            "cod_modulo",
            "eje_potencial__c",
        ]
    ],
    on="id_cliente",
    # how='outer',  # Este es un merge 'outer', para conservar todos los clientes de ambos DataFrames.
    how="inner",
    suffixes=(
        "_df1",
        "_df2",
    ),  # Añadimos sufijos para diferenciar las columnas que se solapan
)

# Reemplazar los valores nulos de 'cod_ruta_df2' con los valores de 'cod_ruta_df1', y viceversa
# Si 'cod_ruta_df2' es nulo, se usará el valor de 'cod_ruta_df1'. Si ambos son nulos, se mantendrá nulo.
df_merged["cod_ruta"] = df_merged["cod_ruta_df2"].combine_first(
    df_merged["cod_ruta_df1"]
)
df_merged["cod_modulo"] = df_merged["cod_modulo_df2"].combine_first(
    df_merged["cod_modulo_df1"]
)

# Eliminar las columnas innecesarias ('cod_ruta_df1', 'cod_ruta_df2', 'cod_modulo_df1', 'cod_modulo_df2')
df_merged = df_merged.drop(
    columns=["cod_ruta_df1", "cod_ruta_df2", "cod_modulo_df1", "cod_modulo_df2"]
)

In [39]:
pan_ventas = df_merged.copy()
# Liberar memoria del df_merged
df_merged = None 
gc.collect()

1566

In [40]:
pan_ventas["cod_cliente"] = pan_ventas["cod_cliente"].astype(str)
pan_ventas["cod_ruta"] = pan_ventas["cod_ruta"].astype(int)
pan_ventas["cod_modulo"] = pan_ventas["cod_modulo"].astype(int)

In [41]:
# Al cruzar visita con ventas. hay clientes de. visita que no tienen informacion de venta, esto se remplaza a nivel de ruta.
# si la ruta entera no tiene ventas, se ignora
# Este df guarda clientes con visitas y sin ventas para luego recomendarlos por separado
df_clientes_con_visitas_sin_ventas = pd.DataFrame()
for ruta in pan_ventas["cod_ruta"].unique():
    temp_df = pan_ventas[pan_ventas["cod_ruta"] == ruta]
    # Si la ruta entera no tiene ninguna venta, se guarda en un dataframe aparte
    if len(temp_df) == (temp_df["cod_articulo_magic"].isnull().sum()):
        df_clientes_con_visitas_sin_ventas = pd.concat(
            [df_clientes_con_visitas_sin_ventas, temp_df], axis=0
        ).reset_index(drop=True)

In [42]:
pan_ventas["id_cliente"].nunique()

50333

In [43]:
PS_VAL_total_clientes_merged = pan_ventas["id_cliente"].nunique()
PS_VAL_total_clientes_merged

50333

In [44]:
PS_VAL_total_skus_merged = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_merged

119

In [45]:
PS_VAL_total_fechas_unicas = pan_ventas.fecha_liquidacion.sort_values().nunique()
PS_VAL_total_fechas_unicas

333

In [46]:
pan_ventas.groupby("cod_sucursal").agg(
    {"cod_articulo_magic": "nunique", "id_cliente": "nunique", "cod_ruta": "nunique"}
)

,cod_articulo_magic,id_cliente,cod_ruta
cod_sucursal,,,
2,114,9111,39
8,111,7457,30
15,115,10524,39
16,106,5856,20
17,108,3446,14
26,109,6733,20
27,110,7208,21


# 2 Ajustar DF de clientes: Se filtrará a los clientes que tienen visita al día siguiente

In [47]:
# CAMBIAR DEPORADE POR SPORADE
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].str.strip()
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].replace({"DEPORADE": "SPORADE"})

pan_prod = pd.read_csv("Input/D_pan_masterProd.csv")

# PAN SEGMENTOS
pan_ventas = pan_ventas.rename(columns={"eje_potencial__c": "new_segment"})
# mapear valores
mapping = {
    "S1": "BLINDAR",
    "S2": "DESARROLLAR",
    "S4": "MANTENER",
    "S5": "OPTIMIZAR",
}

pan_ventas["new_segment"] = pan_ventas["new_segment"].map(mapping)
pan_ventas["new_segment"] = pan_ventas["new_segment"].fillna("OPTIMIZAR")

In [48]:
pan_ventas = pd.merge(
    pan_ventas,
    pan_prod[["cod_articulo_magic", "desc_articulo"]],
    how="left",
    on="cod_articulo_magic",
)

pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"}, inplace=True)
pan_ventas["mes"] = pd.to_datetime(pan_ventas["fecha_liquidacion"]).dt.strftime(
    "%Y-%m-01"
)

In [49]:
pan_ventas.to_parquet("Processed/ventas_12m.parquet", index=False)

## 2.4 Filtrar clientes a visitar mañana

In [50]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [51]:
# Filtrar ventas mayores a 0
# pan_ventas=pan_ventas[pan_ventas["imp_netovta"]>0]
# En lugar de omitir estos clientes, mejor los consideramos como OPTIMIZAR
# pan_ventas.loc[pan_ventas["imp_netovta"] <= 0, "new_segment"] = "OPTIMIZAR"

In [52]:
PS_VAL_total_clientes_imp_netovta_positivo = pan_ventas.id_cliente.nunique()
PS_VAL_total_clientes_imp_netovta_positivo

50333

In [53]:
pan_ventas.dias_de_visita__c.unique()

array(['1;4', '3;7', '3', '3;6', '1', '7', '6', '4', '2;5', '5', '2'],
      dtype=object)

In [54]:
pan_ventas.groupby("dias_de_visita__c").id_cliente.nunique()

dias_de_visita__c
1      4056
1;4    8547
2         6
2;5       6
3      9168
3;6    8363
3;7    7943
4      3558
5         2
6      4232
7      4452
Name: id_cliente, dtype: int64

In [55]:
data_test = (
    pan_ventas[
        ["id_cliente", "dias_de_visita__c", "periodo_de_visita__c", "ultima_visita"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
data_test["ultima_visita"] = pd.to_datetime(
    data_test["ultima_visita"], format="%Y-%m-%d"
)
# Calcula la diferencia en días entre la fecha de cada fila y la fecha de hoy
fecha_actual = datetime.now() + timedelta(days=0)
data_test["dias_pasados"] = (fecha_actual - data_test["ultima_visita"]).dt.days
fecha_actual

datetime.datetime(2026, 3, 30, 21, 37, 46, 615170)

In [56]:
# Obtener el día de la semana actual (1 para lunes, 2 para martes, ..., 7 para domingo)
dia_actual = datetime.now(pytz.timezone("America/Lima")).weekday() + 1

# Si hoy es domingo (7), el día siguiente es lunes (1), de lo contrario, es el siguiente día al actual
if dia_actual == 6:
    dia_siguiente = 7
else:
    dia_siguiente = (dia_actual + 1) % 7
    # dia_siguiente = (dia_actual ) % 7

# Filtrar los clientes que serán visitados mañana
clientes_a_visitar_manana = data_test[
    data_test["dias_de_visita__c"]
    .astype(str)
    .apply(lambda x: str(dia_siguiente) in x.split(";"))
].reset_index(drop=True)
print("dia_actual", dia_actual)
print("dia_siguiente", dia_siguiente)

dia_actual 1
dia_siguiente 2


In [57]:
# Definir condiciones de filtro
condicion_f1 = clientes_a_visitar_manana["periodo_de_visita__c"] == "F1"
condicion_f2 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F2") & (
    clientes_a_visitar_manana["dias_pasados"] > 13
)
condicion_f3 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F3") & (
    clientes_a_visitar_manana["dias_pasados"] > 20
)
condicion_f4 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F4") & (
    clientes_a_visitar_manana["dias_pasados"] > 27
)

# Aplicar las condiciones de filtro
clientes_a_visitar_manana = clientes_a_visitar_manana[
    condicion_f1 | condicion_f2 | condicion_f3 | condicion_f4
].reset_index(drop=True)

In [58]:
clientes_a_visitar_manana.isnull().sum()

id_cliente              0
dias_de_visita__c       0
periodo_de_visita__c    0
ultima_visita           0
dias_pasados            0
dtype: int64

In [59]:
clientes_a_visitar_manana.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [60]:
clientes_a_visitar_manana.dias_de_visita__c.unique()

array(['2;5', '2'], dtype=object)

In [61]:
pan_ventas = pan_ventas[
    pan_ventas["id_cliente"].isin(clientes_a_visitar_manana["id_cliente"])
].reset_index(drop=True)

In [62]:
pan_ventas.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [63]:
pan_ventas.dias_de_visita__c.unique()

array(['2;5', '2'], dtype=object)

In [64]:
PS_VAL_total_clientes_para_manhana = pan_ventas.id_cliente.nunique()
PS_VAL_total_clientes_para_manhana

11

In [65]:
PS_VAL_total_skus_venta_para_manhana = pan_ventas.cod_articulo_magic.nunique()
PS_VAL_total_skus_venta_para_manhana

48

In [66]:
pan_ventas.groupby("cod_sucursal").cod_articulo_magic.nunique()

cod_sucursal
8     12
15    36
16    10
17    17
26    19
Name: cod_articulo_magic, dtype: int64

In [67]:
pan_ventas.to_parquet("Processed/ventas_manana.parquet", index=False)

# 3. Pre Procesamiento

In [68]:
df_ventas = pd.read_parquet("Processed/ventas_manana.parquet")

In [69]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(
    df_ventas["fecha_liquidacion"], format="%Y-%m-%d"
)
df_ventas["desc_marca"] = df_ventas["desc_marca"].str.strip()
df_ventas["desc_categoria"] = df_ventas["desc_categoria"].str.strip()

## 3.1 Mapeo de Pesos por Giro
Los pesos representan el orden en el que se recomendara la marca para cada cliente dependiendo del Giro

In [70]:
# Mapeo de pesos para ordenar las recomendaciones finales segun su importancia por giro
mapeo_pesos = {}
mapeo_diccionario = {}

In [71]:
df_ventas.desc_marca.unique()

array(['SPORADE', 'BIG', 'AJE CIELO', 'CIFRUT', 'PULP', 'D GUSSTO',
       'VOLT GO', 'DILYTE', 'DEST', 'VOLT', 'CIELO', 'BIO ALOE'],
      dtype=object)

In [72]:
df_ventas.desc_categoria.unique()

array(['BEBIDAS DEPORTIVAS', 'GASEOSAS', 'AGUA', 'JUGOS LIGEROS',
       'NECTAR', 'SNACKS', 'ENERGIZANTE', 'SUERO', 'DETERGENTE', 'ARROZ',
       'DETERGENTE EN POLVO', 'ALOE VERA'], dtype=object)

In [73]:
for giro_v in df_ventas["desc_subgiro"].unique():
    temp = df_ventas[(df_ventas["desc_subgiro"] == giro_v)]
    ranks = temp.groupby("desc_categoria")["cant_cajafisicavta"].sum().reset_index()
    ranks.columns = ["index","desc_categoria"]
    ranks = ranks.sort_values(by="desc_categoria", ascending=False)
    if len(list(ranks["desc_categoria"])) <= 5:
        ranks["Ranking"] = range(1, len(ranks) + 1)
    elif len(list(ranks["desc_categoria"])) > 5:
        a = list(ranks["desc_categoria"])
        b = [1, 1, 2, 2]
        # Calculamos el multiplicador para el mapeo de pesos segun la varianza
        if np.std(a) / np.mean(a) <= 1.2:
            multiplicador = 4
        else:
            multiplicador = 2
        if len(a) > 5:
            for i in range(4, len(a)):
                if a[3] <= a[i] * multiplicador:
                    b.append(3)
                else:
                    b.append(3 + i)
        ranks["Ranking"] = b

    print("*" * 20)
    print("GIRO: ", giro_v)
    print("Categorias: ", list(ranks["index"]))
    print("Counts: ", list(ranks["desc_categoria"]))
    mapeo_diccionario[giro_v] = ranks.set_index("index")["Ranking"].to_dict()
    print(ranks.set_index("index")["Ranking"].to_dict())

********************
GIRO:  BODEGAS, TIENDAS
Categorias:  ['BEBIDAS DEPORTIVAS', 'GASEOSAS', 'JUGOS LIGEROS', 'AGUA', 'NECTAR', 'ENERGIZANTE', 'ARROZ', 'SUERO', 'SNACKS', 'DETERGENTE EN POLVO', 'ALOE VERA', 'DETERGENTE']
Counts:  [247.0, 185.0416666666666, 174.5416666666666, 117.5416666666666, 50.0, 20.0, 3.0, 2.8333333333332, 2.0, 2.0, 1.0, 0.0833333333333]
{'BEBIDAS DEPORTIVAS': 1, 'GASEOSAS': 1, 'JUGOS LIGEROS': 2, 'AGUA': 2, 'NECTAR': 7, 'ENERGIZANTE': 8, 'ARROZ': 9, 'SUERO': 10, 'SNACKS': 11, 'DETERGENTE EN POLVO': 12, 'ALOE VERA': 13, 'DETERGENTE': 14}


## 3.2. Filtro de SKU por Ruta
Cada Ruta tiene una cantidad de SKUs distintos, la recomendacion será distinta para cada ruta, sin embargo, aquellas rutas que tengan pocos SKUs serán tratadas como una sola ruta

In [74]:
rutas = (
    df_ventas.groupby(["cod_ruta"])["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()["cod_ruta"]
    .unique()
)
rutas

array([1106, 1115, 1136, 1138, 1181, 1196, 1303, 1310, 1354, 1373, 1378])

In [75]:
low_sku_ruta = []

**Para rutas que tengan más de 45 SKUS**

In [76]:
for ruta in rutas:
    print("*" * 21)
    print("Ruta:", ruta)
    temp = df_ventas[(df_ventas["cod_ruta"] == ruta)]
    print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
    # print(f"Giros en ruta {ruta}:")
    # print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
    if temp["cod_articulo_magic"].nunique() <= 5:
        low_sku_ruta.append(ruta)
    else:
        temp.to_csv(f"Processed/rutas/D_{ruta}_ventas.csv", index=False)

*********************
Ruta: 1106
SKUs disponibles: 9
*********************
Ruta: 1115
SKUs disponibles: 1
*********************
Ruta: 1136
SKUs disponibles: 11
*********************
Ruta: 1138
SKUs disponibles: 9
*********************
Ruta: 1181
SKUs disponibles: 25
*********************
Ruta: 1196
SKUs disponibles: 29
*********************
Ruta: 1303
SKUs disponibles: 8
*********************
Ruta: 1310
SKUs disponibles: 12
*********************
Ruta: 1354
SKUs disponibles: 3
*********************
Ruta: 1373
SKUs disponibles: 8
*********************
Ruta: 1378
SKUs disponibles: 2


**Para rutas que tengan menos de 45 SKUS**

In [77]:
print("*" * 21)
print("Rutas:", low_sku_ruta)
temp = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
# print(f"Giros en ruta {ruta}:")
# print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
temp.to_csv("Processed/rutas/D_low_ruta_ventas.csv", index=False)

*********************
Rutas: [1115, 1354, 1378]
SKUs disponibles: 6


# 4. Modelo Pedido Sugerido

## 4.1. Importando librerias necesarias

In [78]:
import os

from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType
from pyspark.sql.functions import (
    when,
    col,
    regexp_replace,
    concat,
    countDistinct,
    lit,
    monotonically_increasing_id,
    hash,
)

from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [79]:
!java -version

openjdk version "11.0.30" 2026-01-20 LTS
OpenJDK Runtime Environment Corretto-11.0.30.7.1 (build 11.0.30+7-LTS)
OpenJDK 64-Bit Server VM Corretto-11.0.30.7.1 (build 11.0.30+7-LTS, mixed mode)


In [80]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

## 4.2 Construccion del algoritmo ALS

In [81]:
# Se generan csv en Output/Panama/4-resultados/
def als_training_job(cedi, sku_len):

    spark = (
        SparkSession.builder.config("spark.memory.offHeap.enabled", "true")
        .config("spark.memory.offHeap.size", "10G")
        .appName("K-Means for ALS")
        .getOrCreate()
    )

    ventas = (
        spark.read.format("csv")
        .options(header=True, delimiter=",")
        .load(f"Processed/rutas/D_{cedi}_ventas.csv")
    )

    ventas = ventas.na.drop(subset=["fecha_liquidacion"])
    ventas = ventas.groupBy(
        ["id_cliente", "cod_articulo_magic", "cod_compania", "cod_cliente"]
    ).agg(countDistinct("fecha_liquidacion"))
    ventas = ventas.select(
        col("id_cliente").alias("id_cliente"),
        col("count(fecha_liquidacion)").alias("frecuencia"),
        col("cod_articulo_magic").alias("cod_articulo_magic"),
        col("cod_compania").alias("cod_compania"),
        col("cod_cliente").alias("cod_cliente"),
    )

    input_cols = ["frecuencia"]
    vec_assembler = VectorAssembler(inputCols=input_cols, outputCol="features")
    ventas_kmeans = vec_assembler.transform(ventas)
    ventas_kmeans = ventas_kmeans.fillna(0.0, subset=["frecuencia"])

    kmeans = KMeans(featuresCol="features", k=5)
    model = kmeans.fit(ventas_kmeans)

    resumen_ventas = model.transform(ventas_kmeans).groupby("prediction").max()
    resumen_ventas.orderBy(["max(frecuencia)"], ascending=0)
    resumen_ventas = resumen_ventas.select(
        col("prediction").alias("prediction"), col("max(frecuencia)").alias("max")
    )

    maxes = resumen_ventas.select("max").toPandas()["max"]
    maxes = maxes.astype(float)
    maxes = list(maxes)

    labels = [5, 4, 3, 2, 1]
    rank_map = dict(zip(labels, maxes))

    ventas = ventas.withColumn("frecuencia", col("frecuencia").cast(DoubleType()))
    ventas = ventas.withColumn(
        "rating",
        when(ventas["frecuencia"] < rank_map.get(labels[-1]), labels[-1])
        .when(ventas["frecuencia"] < rank_map.get(labels[-2]), labels[-2])
        .when(ventas["frecuencia"] < rank_map.get(labels[-3]), labels[-3])
        .when(ventas["frecuencia"] < rank_map.get(labels[-4]), labels[-4])
        .otherwise(labels[-5]),
    )

    cols = ["id_cliente", "cod_compania", "cod_cliente", "cod_articulo_magic", "rating"]
    als_records = ventas.select(*cols)

    als_records = als_records.withColumn(
        "clienteId", concat("cod_compania", "cod_cliente")
    )
    als_records = als_records.withColumn(
        "clienteId", col("clienteId").cast(IntegerType())
    )
    als_records = als_records.withColumn(
        "cod_articulo_magic", col("cod_articulo_magic").cast(IntegerType())
    )

    als_records = als_records.na.drop(subset=["clienteId", "cod_articulo_magic"])

    input_cols = ["ClienteId", "cod_articulo_magic", "rating"]
    als_records = als_records.select(*input_cols)

    als = ALS(
        rank=15,
        maxIter=5,
        implicitPrefs=True,
        ratingCol="rating",
        itemCol="cod_articulo_magic",
        userCol="ClienteId",
    )
    model = als.fit(als_records)

    # obtener matrices de descomposicion (usuario y articulos)
    user_matrix = model.userFactors
    item_matrix = model.itemFactors

    user_matrix_pd = user_matrix.toPandas()
    item_matrix_pd = item_matrix.toPandas()

    recs = model.recommendForAllUsers(sku_len)
    recs = recs.select("ClienteId", "recommendations.cod_articulo_magic")
    recs_to_parse = recs.toPandas()
    # print(recs_to_parse)

    lista_rec = [f"r{i+1}" for i in range(sku_len)]

    recs_to_parse[lista_rec] = pd.DataFrame(
        recs_to_parse["cod_articulo_magic"].tolist(), index=recs_to_parse.index
    )
    client_recs = pd.melt(
        recs_to_parse, id_vars=["ClienteId"], value_vars=lista_rec
    )

    client_recs["compania"] = client_recs["ClienteId"].apply(lambda x: str(x)[:2])
    client_recs["cliente"] = client_recs["ClienteId"].apply(lambda x: str(x)[2:])
    client_recs["compania"] = client_recs["compania"].apply(lambda x: x.rjust(4, "0"))
    client_recs["id_cliente"] = client_recs["compania"] + "|" + client_recs["cliente"]
    client_recs["cod_articulo_magic"] = client_recs["value"]

    client_recs = (
        client_recs[["id_cliente", "cod_articulo_magic"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Obtener el nombre del archivo CSV
    nombre_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo ya existe
    try:
        # Intentar abrir el archivo para lectura
        with open(nombre_archivo, "r") as f:
            # Si el archivo existe, header=False para no escribir cabeceras
            client_recs.to_csv(
                nombre_archivo, mode="a", header=False, index=False, sep=";"
            )
    except FileNotFoundError:
        # Si el archivo no existe, header=True para escribir las cabeceras
        client_recs.to_csv(nombre_archivo, mode="a", header=True, index=False, sep=";")

    # client_recs.to_csv("Output/Panama/rutas/D_rutas_rec.csv", mode='a', header=False, index=False,sep=";")

## 4.3 Corriendo PS

In [82]:
def runALS():
    # Ruta del archivo que deseas verificar y eliminar si existe
    ruta_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo existe
    if os.path.exists(ruta_archivo):
        # Eliminar el archivo si existe
        os.remove(ruta_archivo)
        print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
    else:
        print(f"El archivo en la ruta {ruta_archivo} no existe.")
    # CORRIENDO ALS PARA RUTAS DE MAS DE 45 SKUs
    for ruta in rutas:
        print("*" * 21)
        temp = pd.read_csv(f"Processed/rutas/D_{ruta}_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en ruta {ruta}:", sku_len)
        als_training_job(ruta, sku_len)
    # CORRIENDO ALS PARA rutas que tengan menos de 45 SKUS
    if len(low_sku_ruta) != 0:
        temp = pd.read_csv(f"Processed/rutas/D_low_ruta_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en low ruta {low_sku_ruta}:", sku_len)
        als_training_job("low_ruta", sku_len)
    else:
        print("No existen rutas con pocos skus :D")

In [83]:
rutas = rutas[~pd.Series(rutas).isin(pd.Series(low_sku_ruta))]
rutas

array([1106, 1136, 1138, 1181, 1196, 1303, 1310, 1373])

In [84]:
runALS()

El archivo en la ruta Output/D_rutas_rec.csv ha sido eliminado.
*********************
SKUs disponibles en ruta 1106: 9


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/30 21:37:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 21:38:00 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/30 21:38:00 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
26/03/30 21:38:00 WARN InstanceBuilder$NativeLAPACK: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


*********************
SKUs disponibles en ruta 1136: 11
*********************
SKUs disponibles en ruta 1138: 9
*********************
SKUs disponibles en ruta 1181: 25
*********************
SKUs disponibles en ruta 1196: 29
*********************
SKUs disponibles en ruta 1303: 8
*********************
SKUs disponibles en ruta 1310: 12
*********************
SKUs disponibles en ruta 1373: 8
SKUs disponibles en low ruta [1115, 1354, 1378]: 6


# 5. Ajustes al output del algoritmo ALS

## 5.-9 Quedarnos con skus que hayan tenido ventas en los ultimos 30 dias

In [85]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [86]:
pan_rec.nunique()

id_cliente            11
cod_articulo_magic    48
dtype: int64

In [87]:
PS_VAL_promedio_recs_por_cliente_modelo = (
    pan_rec.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_modelo

11.727272727272727

In [88]:
pan_rec.id_cliente.nunique()

11

In [89]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

11.727272727272727

In [90]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [91]:
df_ventas = pd.read_parquet("Processed/ventas_manana.parquet")

In [92]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(df_ventas["fecha_liquidacion"])

In [93]:
# Definir el umbral de tiempo (últimos 30 días desde hoy)
# bajamos de 30 a 14 dias 2025-07-25
hoy = pd.Timestamp.today()
fecha_limite = hoy - pd.Timedelta(days=14)
print(fecha_limite)
# Filtrar ventas en los últimos 30 días
ventas_filtradas = df_ventas[df_ventas["fecha_liquidacion"] >= fecha_limite]

2026-03-16 21:38:21.952049


In [94]:
# Obtener los productos vendidos en cada ruta
productos_por_ruta = ventas_filtradas.groupby("cod_ruta")["cod_articulo_magic"].unique().reset_index()

In [95]:
# Unir con los productos vendidos en cada ruta
recomendaciones_validas = pan_rec.merge(productos_por_ruta, on="cod_ruta", how="inner")

In [96]:
recomendaciones_validas.columns = ['id_cliente', 'cod_articulo_magic', 'cod_ruta','lista_sku_ruta']

In [97]:
recomendaciones_validas.shape

(120, 4)

In [98]:
recomendaciones_validas  = recomendaciones_validas[recomendaciones_validas.apply(lambda row: row["cod_articulo_magic"] in row["lista_sku_ruta"], axis=1)].reset_index(drop=True)

In [99]:
recomendaciones_validas.shape

(60, 4)

In [100]:
recomendaciones_validas.head()

,id_cliente,cod_articulo_magic,cod_ruta,lista_sku_ruta
0,CAM|0076|326385,598820,1106,"[523689, 598823, 598820, 598821, 500041, 52423..."
1,CAM|0076|326385,500041,1106,"[523689, 598823, 598820, 598821, 500041, 52423..."
2,CAM|0076|326385,598821,1106,"[523689, 598823, 598820, 598821, 500041, 52423..."
3,CAM|0076|326385,522922,1106,"[523689, 598823, 598820, 598821, 500041, 52423..."
4,CAM|0076|326385,598823,1106,"[523689, 598823, 598820, 598821, 500041, 52423..."


In [101]:
pan_rec = recomendaciones_validas[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [102]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0076|326385,598820
1,CAM|0076|326385,500041
2,CAM|0076|326385,598821
3,CAM|0076|326385,522922
4,CAM|0076|326385,598823


In [103]:
pan_rec.shape

(60, 2)

In [104]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-8 Recomendar de acuerdo a Subida, Bajada, Mantener

In [105]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
#pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [106]:
pan_rec.shape

(60, 2)

In [107]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [108]:
def desplazar_nulos_fila(fila):
    return pd.Series(sorted(fila, key=pd.isna), index=fila.index)

In [109]:
# Definir una función para asignar los valores subida, mantener y bajar
def clasificar_valor(x):
    if x > 0:
        return "S"
    elif x == 0:
        return "M"
    else:
        return "B"

In [110]:
df_ventas['fecha_liquidacion'] = pd.to_datetime(df_ventas['fecha_liquidacion'])

# Definir la fecha de referencia (hoy)
fecha_actual = datetime.now(pytz.timezone("America/Lima"))#.strftime('%Y-%m-%d')
fecha_30dias_atras = (fecha_actual - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
fecha_60dias_atras = (fecha_actual - pd.Timedelta(days=60)).strftime('%Y-%m-%d')

print("fecha_actual",fecha_actual)
print("fecha_30dias_atras",fecha_30dias_atras)
print("fecha_60dias_atras",fecha_60dias_atras)

fecha_actual 2026-03-30 16:38:22.062860-05:00
fecha_30dias_atras 2026-02-28
fecha_60dias_atras 2026-01-29


In [111]:
# Filtrar los últimos 30 días
df_ultimos_30 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_30dias_atras) 
                          & (df_ventas['fecha_liquidacion'] <= fecha_actual.strftime('%Y-%m-%d'))]

# Filtrar los días 31 a 60
df_31_60 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_60dias_atras)
              & (df_ventas['fecha_liquidacion'] <= fecha_30dias_atras)]

# Agrupar por codigo_usuario, ruta, codigo_producto y sumar ventas
ventas_ultimos_30 = df_ultimos_30.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_ultimos_30["mes"] = "0_30"
ventas_31_60 = df_31_60.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_31_60["mes"] = "31_60"

df_grouped = pd.concat([ventas_ultimos_30,ventas_31_60],axis=0,ignore_index=True)
df_grouped = pd.pivot_table(
    df_grouped,
    values="imp_netovta",
    index=["cod_ruta", "cod_articulo_magic"],
    columns=["mes"],
    aggfunc="sum",
).reset_index()

# Rellenando con 0 si es nulo
df_grouped["0_30"] = df_grouped["0_30"].fillna(0)
df_grouped["31_60"] = df_grouped["31_60"].fillna(0)

# Asignando variacion de productos por mes
df_grouped["v1_2"] = (
    (df_grouped[df_grouped.columns[2]] - df_grouped[df_grouped.columns[3]])
    / df_grouped[df_grouped.columns[3]]
    * 100
)
df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)
df_grouped["v1_2"] = df_grouped["v1_2"].fillna(-1)
df_grouped["vp"] = (df_grouped["v1_2"])

/tmp/ipykernel_21325/261865676.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)


In [112]:
# Aplicar la función a la columna 'valor' y guardar el resultado en una nueva columna 'flag'
df_grouped["flag"] = df_grouped["vp"].apply(lambda x: clasificar_valor(x))
# Definir el mapeo de valores
mapeo_flag = {"S": 0, "M": 1, "B": 2}
# Aplicar el mapeo a la columna 'clasificacion' y guardar el resultado en una nueva columna 'flag_rank'
df_grouped["flag_rank"] = df_grouped["flag"].map(mapeo_flag)

In [113]:
df_grouped

mes,cod_ruta,cod_articulo_magic,0_30,31_60,v1_2,vp,flag,flag_rank
0,1106,500041,41.9643,0.0,-1.0,-1.0,B,2
1,1106,522918,69.6429,0.0,-1.0,-1.0,B,2
2,1106,522922,27.6786,0.0,-1.0,-1.0,B,2
3,1106,523689,53.5714,0.0,-1.0,-1.0,B,2
4,1106,523845,22.3214,0.0,-1.0,-1.0,B,2
...,...,...,...,...,...,...,...,...
94,1373,523374,20.5357,0.0,-1.0,-1.0,B,2
95,1373,523533,83.0358,0.0,-1.0,-1.0,B,2
96,1373,523689,8.9286,0.0,-1.0,-1.0,B,2
97,1378,522918,139.2857,0.0,-1.0,-1.0,B,2


In [114]:
pan_rec.shape

(60, 3)

In [115]:
pan_rec = pd.merge(pan_rec,df_grouped[["cod_ruta", "cod_articulo_magic","flag_rank"]],on=["cod_ruta", "cod_articulo_magic"],how="left")

In [116]:
pan_rec["flag_rank"] = pan_rec["flag_rank"].fillna(3)

In [117]:
# Agregar una columna con el orden original
pan_rec["original_order"] = pan_rec.index

In [118]:
# Ordenar por cod_cliente, rank_producto (ascendente) y mantener el orden original en caso de empate
pan_rec = pan_rec.sort_values(by=["id_cliente", "flag_rank", "original_order"], ascending=[True, True, True])#.drop(columns=["original_order"])

In [119]:
pan_rec.head()

,id_cliente,cod_articulo_magic,cod_ruta,flag_rank,original_order
11,CAM|0076|322102,523271,1181,0,11
13,CAM|0076|322102,598822,1181,0,13
19,CAM|0076|322102,599266,1181,0,19
21,CAM|0076|322102,511908,1181,0,21
12,CAM|0076|322102,522918,1181,2,12


In [120]:
pan_rec.shape

(60, 5)

In [121]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [122]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0076|322102,523271
1,CAM|0076|322102,598822
2,CAM|0076|322102,599266
3,CAM|0076|322102,511908
4,CAM|0076|322102,522918


In [123]:
pan_rec.shape

(60, 2)

In [124]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-7 Usamos el archivo de validacion para no recomendar SKUs que no se deben a las rutas

In [125]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [126]:
pan_rec.shape

(60, 2)

In [127]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [128]:
pan_rec.isnull().sum()

id_cliente            0
cod_articulo_magic    0
cod_ruta              0
dtype: int64

In [129]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

cod_ruta
1106     9
1115     1
1136     2
1181    11
1196    10
Name: cod_articulo_magic, dtype: int64

In [130]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

6.0

In [131]:
# Ruta del archivo en S3
s3_path = "s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/cam/maestro_productos_cam000"

# Leer el CSV directamente a un DataFrame
skus_val = pd.read_csv(s3_path,sep = ";")
# skus_val = pd.read_csv("maestra_productos_cam_20250217.csv",sep=";")
# skus_val.head()

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [132]:
skus_val = skus_val[skus_val.cod_compania==76]
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print("fecha_proceso: ", hoy)
# Filtrar solo las filas con la fecha de hoy
# skus_val = skus_val[skus_val['fecha_proceso'] == hoy]
skus_val["cod_compania"] = (skus_val["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
skus_val["id_cliente"] = (
    "CAM"
    + "|"
    + skus_val["cod_compania"].astype(str)
    + "|"
    + skus_val["cod_cliente"].astype(str)
)

fecha_proceso:  20260330


In [133]:
skus_val.shape

(1378776, 14)

In [134]:
# skus_val = pd.read_csv("productos_pedido_sugerido_validacion_cam_090225.csv",sep=";")
# skus_val = skus_val[skus_val.cod_compania==70].reset_index(drop=True)

In [135]:
# Cada ruta pertenece a una compañia
skus_val.groupby("cod_ruta").cod_compania.nunique().mean()

1.0

In [136]:
pan_rec = pd.merge(pan_rec,skus_val[['cod_articulo_magic', 'id_cliente']].drop_duplicates(),left_on=["id_cliente","cod_articulo_magic"],right_on=["id_cliente","cod_articulo_magic"],how="inner")

In [137]:
pan_rec.shape

(60, 3)

In [138]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

cod_ruta
1106     9
1115     1
1136     2
1181    11
1196    10
Name: cod_articulo_magic, dtype: int64

In [139]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

6.0

In [140]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

6.0

In [141]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]]

In [142]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,CAM|0076|322102,523271
1,CAM|0076|322102,598822
2,CAM|0076|322102,599266
3,CAM|0076|322102,511908
4,CAM|0076|322102,522918


In [143]:
pan_rec.shape

(60, 2)

In [144]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-6 Priorizar/mover SKUs para asegurar su recomendacion (Solo correr si se tienen SKUs a priorizar)

In [145]:
# pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "CAM|" + pan_rec["id_cliente"]

In [146]:
# skus_a_priorizar = [500580, 500713, 501022, 501505, 501603, 502501, 522398, 622061]

In [147]:
# pan_rec.shape

In [148]:
# # SKUs priorizados
# temp_1 = pan_rec[pan_rec.cod_articulo_magic.isin(skus_a_priorizar)]
# # SKUs NO priorizados
# temp_2 = pan_rec[~pan_rec.cod_articulo_magic.isin(skus_a_priorizar)]

In [149]:
# pan_rec = pd.concat([temp_1, temp_2], axis=0, ignore_index=True)

In [150]:
# pan_rec.shape

In [151]:
# pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-5 Filtrar SKUs por STOCK

In [152]:
stock = pd.read_csv("s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/cam/D_stock_gt.csv")

In [153]:
stock.drop(columns=["Fecha","Database"], inplace=True)
stock.columns=["cod_compania","cod_sucursal","cod_articulo_magic","stock_cf"]

In [154]:
stock["cod_compania"] = (stock["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
stock["cod_sucursal"] = (stock["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [155]:
fecha_hace_12_dias = (datetime.today() - timedelta(days=12)).strftime('%Y-%m-%d')
print(fecha_hace_12_dias)

2026-03-18


In [156]:
# Promedio diario de cajas fisicas por sucursal y compañia de los ultimos 12 dias
prom_diario_vta = df_ventas[df_ventas.cant_cajafisicavta > 0]
prom_diario_vta = df_ventas[df_ventas.fecha_liquidacion>=fecha_hace_12_dias].groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic","fecha_liquidacion"]).cant_cajafisicavta.sum().reset_index().groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic"]).cant_cajafisicavta.mean().reset_index()
prom_diario_vta["cod_compania"] = (prom_diario_vta["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
prom_diario_vta["cod_sucursal"] = (prom_diario_vta["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [157]:
df_stock = pd.merge(prom_diario_vta,stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"],how="left")
df_stock["dias_stock"] = df_stock["stock_cf"]/df_stock["cant_cajafisicavta"]
df_stock =  df_stock[(df_stock.dias_stock > 3) & (df_stock.cant_cajafisicavta > 0)].reset_index(drop=True)

In [158]:
pan_rec.nunique()

id_cliente            10
cod_articulo_magic    33
dtype: int64

In [159]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

6.0

In [160]:
pan_rec.shape

(60, 2)

In [161]:
# Cruzamos stock

In [162]:
test = pd.merge(pan_rec,df_ventas[["id_cliente","cod_compania","cod_sucursal"]].drop_duplicates(),on="id_cliente",how="left")

In [163]:
test["cod_compania"] = (test["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
test["cod_sucursal"] = (test["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [164]:
test_f = pd.merge(test, df_stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"], how="inner")

In [165]:
test_f.head()

,id_cliente,cod_articulo_magic,cod_compania,cod_sucursal,cant_cajafisicavta,stock_cf,dias_stock
0,CAM|0076|322102,523271,0076,15,3.500000,4232,1209.142857
1,CAM|0076|322102,598822,0076,15,2.000000,1062,531.000000
2,CAM|0076|322102,599266,0076,15,6.000000,856,142.666667
3,CAM|0076|322102,511908,0076,15,3.000000,4753,1584.333333
4,CAM|0076|322102,522918,0076,15,5.333333,9928,1861.500000


In [166]:
test_f.shape

(51, 7)

In [167]:
test_f.nunique()

id_cliente             9
cod_articulo_magic    29
cod_compania           1
cod_sucursal           4
cant_cajafisicavta    13
stock_cf              47
dias_stock            47
dtype: int64

In [168]:
PS_VAL_promedio_recs_por_cliente_stock = (
    test_f.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_stock

5.666666666666667

In [169]:
pan_rec = test_f[["id_cliente","cod_articulo_magic"]].copy()

In [170]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-4 Mantener Solo SKUS en especifico (no correr sin CSV o XLSX de SKUs activos)

In [171]:
# pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# # pan_rec["id_cliente"]="CAM|"+pan_rec["id_cliente"]

In [172]:
# sku_con_precio = pd.read_excel(
#     "Input/CR - SKUS ACTIVOS_2024-10-31.xlsx", sheet_name="GTM CR - PRODUCTOS"
# )

In [173]:
# pan_rec = pan_rec[
#     pan_rec.cod_articulo_magic.isin(sku_con_precio["SKU"].unique())
# ].reset_index(drop=True)
# pan_rec

In [174]:
# # Size de recomendaciones luego de filtar solo a los del excel de SKUs activos
# pan_rec.shape

In [175]:
# PS_VAL_total_skus_recomendados_stock_y_activos = pan_rec["cod_articulo_magic"].nunique()
# PS_VAL_total_skus_recomendados_stock_y_activos

In [176]:
# pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-3 Quitar Recomendaciones de SKUS en especifico (no correr sin tener SKUs especificos)

In [177]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [178]:
# 622402 no deja vender por unidades
skus_sin_precio = []

In [179]:
pan_rec = pan_rec[~(pan_rec["cod_articulo_magic"].isin(skus_sin_precio))].reset_index(
    drop=True
)

In [180]:
# PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda = (
#     pan_rec.cod_articulo_magic.nunique()
#  )
# PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda

In [181]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-2 Quitar Recomendaciones de los ultimos 2 semanas (no correr sin historial)

In [182]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"]="CAM|"+pan_rec["id_cliente"]

In [183]:
pan_rec

,id_cliente,cod_articulo_magic
0,CAM|0076|322102,523271
1,CAM|0076|322102,598822
2,CAM|0076|322102,599266
3,CAM|0076|322102,511908
4,CAM|0076|322102,522918
5,CAM|0076|322102,598823
6,CAM|0076|322102,500179
7,CAM|0076|322102,523533
8,CAM|0076|322102,524384
9,CAM|0076|322102,524556


In [184]:
master_prod = pd.read_csv("Input/D_pan_masterProd.csv")

In [185]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-31'

In [186]:
ruta_archivo = f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv"
# Verificar si el archivo existe
if os.path.exists(ruta_archivo):
    # Eliminar el archivo si existe
    os.remove(ruta_archivo)
    print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
else:
    print(f"El archivo en la ruta {ruta_archivo} no existe.")

El archivo en la ruta Output/PS_piloto_v1/D_base_pedidos_2026-03-31.csv no existe.


In [187]:
# Directorio donde se encuentran los archivos
directorio = "Output/PS_piloto_v1/"

# Expresión regular para encontrar fechas en el nombre de los archivos
patron_fecha = r"\d{4}-\d{2}-\d{2}"

# Lista para almacenar las fechas extraídas
fechas = []

# Iterar sobre los archivos en el directorio
for archivo in os.listdir(directorio):
    # Verificar si el archivo es un archivo CSV y coincide con el patrón de nombres
    if archivo.endswith(".csv") and re.match(
        r"^D_base_pedidos_\d{4}-\d{2}-\d{2}\.csv$", archivo
    ):
        # Extraer la fecha del nombre del archivo
        fecha = re.search(patron_fecha, archivo).group()
        # Agregar la fecha a la lista
        fechas.append(fecha)

In [188]:
last_7_days_2 = sorted(fechas)[-14:]
last_7_days_2

['2026-03-14',
 '2026-03-16',
 '2026-03-17',
 '2026-03-18',
 '2026-03-19',
 '2026-03-20',
 '2026-03-21',
 '2026-03-23',
 '2026-03-24',
 '2026-03-25',
 '2026-03-26',
 '2026-03-27',
 '2026-03-28',
 '2026-03-30']

In [189]:
if len(last_7_days_2) > 0:
    # Leer recomendaciones de los ultimos 7 dias y cruzar con clientes que tienen recomendacion para mañana
    last_7_days_recs = pd.DataFrame()
    for fecha_rec in last_7_days_2:
        df_temp = pd.read_csv(
            f"Output/PS_piloto_v1/D_base_pedidos_{fecha_rec}.csv",
            dtype={"Compania": "str", "Cliente": "str"},
        )
        df_temp["id_cliente"] = "CAM|" + df_temp["Compania"] + "|" + df_temp["Cliente"]
        df_temp = df_temp[df_temp["id_cliente"].isin(pan_rec["id_cliente"].unique())]
        last_7_days_recs = pd.concat([last_7_days_recs, df_temp], axis=0)
        print(f"{fecha_rec} done")
else:
    last_7_days_recs = []

2026-03-14 done
2026-03-16 done
2026-03-17 done
2026-03-18 done
2026-03-19 done
2026-03-20 done
2026-03-21 done
2026-03-23 done
2026-03-24 done
2026-03-25 done
2026-03-26 done
2026-03-27 done
2026-03-28 done
2026-03-30 done


In [190]:
if len(last_7_days_recs)>0:
    # Utilizamos merge para combinar los dataframes, utilizando la columna 'Cliente' como clave
    df_combinado = pd.merge(
        pan_rec,
        last_7_days_recs,
        left_on=["id_cliente", "cod_articulo_magic"],
        right_on=["id_cliente", "Producto"],
        how="left",
        indicator=True,
    )
    # Filtramos los registros que solo están en el DataFrame 1
    df_resultado = df_combinado[df_combinado["_merge"] == "left_only"][
        ["id_cliente", "cod_articulo_magic"]
    ].reset_index(drop=True)
    df_resultado
else:
    df_resultado = []

In [191]:
if len(df_resultado)>0:
    df_resultado.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.0 Calcular Irregularidad de clientes

In [192]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [193]:
pan_rec["id_cliente"].nunique()

9

In [194]:
now = pd.to_datetime(datetime.now(pytz.timezone("America/Lima")).strftime("%Y-%m-01"))
fecha_doce_meses_atras = now - pd.DateOffset(months=12)
lista_m12 = [fecha_doce_meses_atras + pd.DateOffset(months=i) for i in range(12)]
lista_m12

fecha_seis_meses_atras = now - pd.DateOffset(months=6)
lista_m6 = [fecha_seis_meses_atras + pd.DateOffset(months=i) for i in range(6)]
lista_m6

[Timestamp('2025-09-01 00:00:00'),
 Timestamp('2025-10-01 00:00:00'),
 Timestamp('2025-11-01 00:00:00'),
 Timestamp('2025-12-01 00:00:00'),
 Timestamp('2026-01-01 00:00:00'),
 Timestamp('2026-02-01 00:00:00')]

In [195]:
temp_df_ventas = df_ventas[df_ventas["id_cliente"].isin(pan_rec["id_cliente"].unique())]

In [196]:
qw = (
    temp_df_ventas[["id_cliente", "mes"]]
    .drop_duplicates()
    .sort_values(["id_cliente", "mes"])
    .groupby("id_cliente")
    .tail(12)
    .reset_index(drop=True)
)
qw["mes"] = pd.to_datetime(qw["mes"])
qw["m12"] = qw["mes"].isin(lista_m12)
qw["m6"] = qw["mes"].isin(lista_m6)

In [197]:
categoria_cliente = qw.groupby("id_cliente")[["m12", "m6"]].sum().reset_index()

In [198]:
# Definir condiciones
condicion_super = categoria_cliente["m12"] == 12
condicion_frecuente = (
    (categoria_cliente["m12"] < 12)
    & (categoria_cliente["m12"] >= 6)
    & (categoria_cliente["m6"] == 6)
)
condicion_regulares = (categoria_cliente["m6"] <= 6) & (categoria_cliente["m12"] <= 6)
condicion_riesgo = (categoria_cliente["m6"] < 6) & (categoria_cliente["m6"] >= 4)
condicion_irregulares = categoria_cliente["m6"] < 4

# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_super, "categoria_cliente"] = "Super"
categoria_cliente.loc[condicion_frecuente, "categoria_cliente"] = "Frecuente"
categoria_cliente.loc[condicion_regulares, "categoria_cliente"] = "Regular"
categoria_cliente.loc[condicion_riesgo, "categoria_cliente"] = "Riesgo"
categoria_cliente.loc[condicion_irregulares, "categoria_cliente"] = "Irregular"

**Recalcular Irregularidad**

In [199]:
# Definir condiciones
condicion_regular = (
    (categoria_cliente["categoria_cliente"] == "Super")
    | (categoria_cliente["categoria_cliente"] == "Frecuente")
    | (categoria_cliente["categoria_cliente"] == "Regular")
    | (categoria_cliente["categoria_cliente"] == "Riesgo")
)
condicion_irregular = categoria_cliente["categoria_cliente"] == "Irregular"
# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_regular, "categoria_cliente_2"] = "Regular"
categoria_cliente.loc[condicion_irregular, "categoria_cliente_2"] = "Irregular"

In [200]:
categoria_cliente

,id_cliente,m12,m6,categoria_cliente,categoria_cliente_2
0,CAM|0076|322102,4,4,Riesgo,Regular
1,CAM|0076|323729,2,2,Irregular,Irregular
2,CAM|0076|325499,1,1,Irregular,Irregular
3,CAM|0076|325505,1,1,Irregular,Irregular
4,CAM|0076|326346,0,0,Irregular,Irregular
5,CAM|0076|326366,0,0,Irregular,Irregular
6,CAM|0076|326385,0,0,Irregular,Irregular
7,CAM|0076|326724,0,0,Irregular,Irregular
8,CAM|0076|326743,0,0,Irregular,Irregular


In [201]:
categoria_cliente["categoria_cliente"].value_counts(dropna=False)

categoria_cliente
Irregular    8
Riesgo       1
Name: count, dtype: int64

In [202]:
categoria_cliente["categoria_cliente_2"].value_counts(dropna=False)

categoria_cliente_2
Irregular    8
Regular      1
Name: count, dtype: int64

## 5.1. Obteniendo marcas de recomendaciones por cliente

In [203]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [204]:
pan_ventas = df_ventas[(df_ventas["cod_ruta"].isin(rutas))]
pan_ventas2 = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
pan_ventas = pd.concat([pan_ventas, pan_ventas2], axis=0)

In [205]:
marca_articulo = pan_ventas[["desc_categoria", "cod_articulo_magic"]].drop_duplicates()
print(marca_articulo.shape)
marca_articulo.head()

(48, 2)


,desc_categoria,cod_articulo_magic
0,BEBIDAS DEPORTIVAS,522451
2,GASEOSAS,500179
3,AGUA,523533
5,BEBIDAS DEPORTIVAS,522450
7,JUGOS LIGEROS,523271


In [206]:
cliente_rec_marca = pd.merge(
    pan_rec, marca_articulo, on="cod_articulo_magic", how="left"
)
cliente_rec_marca["desc_categoria"] = cliente_rec_marca["desc_categoria"].str.strip()
print(cliente_rec_marca.shape)
cliente_rec_marca

(44, 3)


,id_cliente,cod_articulo_magic,desc_categoria
0,CAM|0076|322102,523271,JUGOS LIGEROS
1,CAM|0076|322102,598822,NECTAR
2,CAM|0076|322102,599266,ENERGIZANTE
3,CAM|0076|322102,511908,JUGOS LIGEROS
4,CAM|0076|322102,524384,GASEOSAS
5,CAM|0076|322102,524556,AGUA
6,CAM|0076|322102,524237,ENERGIZANTE
7,CAM|0076|323729,511908,JUGOS LIGEROS
8,CAM|0076|323729,500179,GASEOSAS
9,CAM|0076|323729,522918,AGUA


## 5.2. Calcular marcas distintas recomendadas por cliente

In [207]:
# Conteo de categorias recomendadas por Cliente
cods2 = cliente_rec_marca.groupby("id_cliente")["desc_categoria"].nunique().reset_index()
print(cods2.shape)
cods2.head()

(9, 2)


,id_cliente,desc_categoria
0,CAM|0076|322102,5
1,CAM|0076|323729,3
2,CAM|0076|325499,4
3,CAM|0076|325505,4
4,CAM|0076|326346,1


## 5.3 Quitando los SKUs de las ultimas 2 semanas (Evitar recompra)

In [208]:
last_2_weeks = pd.to_datetime(datetime.now(pytz.timezone("America/Lima"))) - pd.DateOffset(
    days=14
)
last_2_weeks

Timestamp('2026-03-16 16:38:45.834539-0500', tz='America/Lima')

In [209]:
# Asegurarse de que 'fecha_liquidacion' esté en formato datetime
pan_ventas["fecha_liquidacion"] = pd.to_datetime(pan_ventas["fecha_liquidacion"])
# Convertir 'fecha_liquidacion' a la misma zona horaria que 'last_year'
pan_ventas["fecha_liquidacion"] = pan_ventas["fecha_liquidacion"].dt.tz_localize(
    "America/Lima", nonexistent="NaT", ambiguous="NaT")

In [210]:
df_quitar_2_weeks = pan_ventas[pan_ventas["fecha_liquidacion"]>=last_2_weeks][["id_cliente","cod_articulo_magic"]].drop_duplicates().reset_index(drop=True)

In [211]:
# Hacemos un merge para identificar coincidencias
cliente_rec_sin4 = cliente_rec_marca.merge(df_quitar_2_weeks, on=['id_cliente', 'cod_articulo_magic'], how='left', indicator=True)
# Filtramos solo los registros que NO están en df_quitar_2_weeks
cliente_rec_sin4 = cliente_rec_sin4[cliente_rec_sin4['_merge'] == 'left_only'].drop(columns=['_merge'])

In [212]:
cliente_rec_marca.id_cliente.nunique()

9

In [213]:
cliente_rec_sin4.id_cliente.nunique()

0

In [214]:
cliente_rec_sin4

,id_cliente,cod_articulo_magic,desc_categoria


## 5.4 Filtro para calcula antiguedad de clientes

In [215]:
pan_ventas = pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"})

In [216]:
# Convertir la columna 'fecha_creacion' a tipo fecha
pan_ventas["fecha_creacion"] = pd.to_datetime(
    pan_ventas["fecha_creacion"].astype(str).str.replace(r"\.", "", regex=True),
    format="%Y%m%d",
    errors="coerce"  # para que los NaN o valores inválidos no generen error
)

In [217]:
# Definir una función para etiquetar los clientes según su fecha de creación
def etiquetar_cliente(fecha_creacion):
    if pd.isnull(fecha_creacion):
        return (
            "nf"  # Si la fecha de creación es nula, etiquetar como 'nf' (no encontrada)
        )
    else:
        hoy = datetime.now()  # Obtener la fecha actual
        hace_12_meses = hoy - timedelta(days=365)  # Calcular hace 12 meses
        if fecha_creacion >= hace_12_meses:
            return "new_client"  # Si la fecha de creación está dentro de los últimos 12 meses, etiquetar como 'new_client'
        else:
            return "old_client"  # Si la fecha de creación es anterior a los últimos 12 meses, etiquetar como 'old_client'

In [218]:
# Aplicar la función a la columna 'fecha_creacion' para crear la nueva columna 'tipo_cliente'
pan_ventas["antiguedad"] = pan_ventas["fecha_creacion"].apply(etiquetar_cliente)

In [219]:
pan_antiguedad_clientes = (
    pan_ventas[["id_cliente", "fecha_creacion", "antiguedad"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

## 5.4.2 Obteniendo datos necesarios para armar DF para subir a SalesForce

In [220]:
datos_para_salesforce = (
    pan_ventas.groupby(
        ["id_cliente", "cod_compania", "cod_sucursal", "cod_cliente", "cod_ruta"]
    )[
        [
            "id_cliente",
            "cod_compania",
            "cod_sucursal",
            "cod_cliente",
            "cod_modulo",
            "cod_ruta",
        ]
    ]
    .head(1)
    .reset_index(drop=True)
)

## 5.5 Juntando las Recomendaciones Finales
 - Ordenado de acuerdo al peso de la recomendación y el id de usuario.
 - Hay 5 skus de 5 marcas distintas que se recomiendan a cada usuario
 - Ningún SKU que haya comprado el usuario en sus ultimas X visitas será recomendado. La cantidad de visitas (X) a ignorar depende de sus visitas promedio mensuales.

In [221]:
# Obtener primer SKU de cada marca de la recomendacion
final_rec = cliente_rec_sin4.groupby(["id_cliente", "desc_categoria"]).first().reset_index()
# Agregar Marca de sus primeras compras
final_rec = pd.merge(final_rec, cods2, how="left", on="id_cliente")
# Agregar descripcion de SKU recomendado
pan_prod = pan_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
pan_prod = pan_prod.groupby(["cod_articulo_magic"]).first().reset_index()
final_rec = pd.merge(final_rec, pan_prod, how="left", on="cod_articulo_magic")
# Cambiar nombre de columnas
final_rec.columns = ["id_cliente", "marca_rec", "sku", "len_marca_rec", "desc_rec"]
# Agregar Giro del cliente
giros = df_ventas[df_ventas["id_cliente"].isin(final_rec["id_cliente"].unique())][
    ["id_cliente", "desc_giro", "desc_subgiro"]
].drop_duplicates()
final_rec = pd.merge(final_rec, giros, on="id_cliente", how="left")
# Agregar si el cliente es Regular o Irregular
final_rec = pd.merge(
    final_rec,
    categoria_cliente[["id_cliente", "categoria_cliente_2"]],
    on="id_cliente",
    how="left",
)
final_rec.loc[final_rec["categoria_cliente_2"] == "Irregular", "desc_giro"] = np.nan
# Agregar antiguedad de cliente
final_rec = pd.merge(final_rec, pan_antiguedad_clientes, how="left", on="id_cliente")
# Agregar segmento de cliente
info_segmentos = pan_ventas[
    [
        "id_cliente",
        "new_segment",
        "dias_de_visita__c",
        "periodo_de_visita__c",
        "ultima_visita",
    ]
].drop_duplicates()
info_segmentos = info_segmentos.groupby(["id_cliente"]).first().reset_index()
final_rec = pd.merge(final_rec, info_segmentos, how="left", on="id_cliente")
# Agregar datos necesarios para SALESFORCE
final_rec = pd.merge(final_rec, datos_para_salesforce, how="left", on="id_cliente")
# Agregar Peso de marcas dependiendo del giro del cliente
final_rec["peso"] = final_rec.apply(
    lambda row: mapeo_diccionario.get(row["desc_subgiro"], {}).get(row["marca_rec"], 5),
    axis=1)
# Ordenar la recomendacion de acuerdo al peso de la marca y nos quedamos con las 5 primeras marcas
final_rec = (
    final_rec.sort_values(["id_cliente", "peso"]).groupby(["id_cliente"]).head(5))
# Agregamos rank de la marca recomendada
final_rec["marca_rec_rank"] = final_rec.groupby("id_cliente").cumcount() + 1
final_rec = final_rec.reset_index(drop=True)

In [222]:
final_rec["sku"] = final_rec["sku"].astype(int)
final_rec["peso"] = final_rec["peso"].astype(int)
final_rec["marca_rec_rank"] = final_rec["marca_rec_rank"].astype(int)

## 5.7 Filtro de Recomendaciones por Segmento de cliente

In [223]:
# Definir una función para filtrar las filas según el valor de 'new_segment'
def filtrar_segmento(group):
    segmento = group["new_segment"].iloc[
        0
    ]  # Obtener el valor del segmento para el grupo
    if segmento == "BLINDAR":
        return group.head(1)  # Si el segmento es 'Blindar', mantener la primera fila
    elif segmento == "MANTENER":
        return group.head(
            2
        )  # Si el segmento es 'Mantener', mantener las dos primeras filas
    elif segmento == "DESARROLLAR":
        return group.head(
            3
        )  # Si el segmento es 'Desarrollar', mantener las tres primeras filas
    elif segmento == "OPTIMIZAR":
        return group.head(
            4
        )  # Si el segmento es 'Optimizar', mantener las cuatro primeras filas
    else:
        return group  # Si el segmento no es ninguno de los especificados, mantener todas las filas

In [224]:
final_rec.nunique()

id_cliente              0
marca_rec               0
sku                     0
len_marca_rec           0
desc_rec                0
desc_giro               0
desc_subgiro            0
categoria_cliente_2     0
fecha_creacion          0
antiguedad              0
new_segment             0
dias_de_visita__c       0
periodo_de_visita__c    0
ultima_visita           0
cod_compania            0
cod_sucursal            0
cod_cliente             0
cod_modulo              0
cod_ruta                0
peso                    0
marca_rec_rank          0
dtype: int64

In [225]:
PS_VAL_promedio_recs_por_cliente_negocio = (
    final_rec.groupby("id_cliente")["sku"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_negocio

nan

In [226]:
final_rec = (
    final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)
)

/tmp/ipykernel_21325/3325175710.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)


In [227]:
final_rec.groupby("new_segment")["id_cliente"].nunique()

Series([], Name: id_cliente, dtype: int64)

In [228]:
final_rec.nunique()

id_cliente              0
marca_rec               0
sku                     0
len_marca_rec           0
desc_rec                0
desc_giro               0
desc_subgiro            0
categoria_cliente_2     0
fecha_creacion          0
antiguedad              0
new_segment             0
dias_de_visita__c       0
periodo_de_visita__c    0
ultima_visita           0
cod_compania            0
cod_sucursal            0
cod_cliente             0
cod_modulo              0
cod_ruta                0
peso                    0
marca_rec_rank          0
dtype: int64

In [229]:
final_rec.groupby("id_cliente")["sku"].nunique().mean()

nan

## 5.8. Guardando recomendacion para D&A y Comercial

In [230]:
final_rec.id_cliente.nunique()

0

In [231]:
final_rec.dias_de_visita__c.value_counts(dropna=False)

Series([], Name: count, dtype: int64)

In [232]:
final_rec.periodo_de_visita__c.value_counts(dropna=False)

Series([], Name: count, dtype: int64)

### 5.8.1 Para D&A

In [233]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-31'

In [234]:
PS_VAL_total_clientes_con_visita_no_venta = final_rec.id_cliente.nunique()
PS_VAL_total_clientes_con_visita_no_venta

0

In [235]:
final_rec["cod_cliente"] = final_rec["id_cliente"].str.split("|").str[-1]

In [236]:
PS_VAL_marcas_recomendadas = final_rec.marca_rec.unique()
PS_VAL_marcas_recomendadas

array([], dtype=object)

In [237]:
final_rec.to_csv(
    f"Output/PS_data_piloto_v1/D_pan_recs_data_{fecha_tomorrow}.csv", index=False
)

In [238]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Guatemala/Output/PS_data_piloto_v1/D_pan_recs_data_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(final_rec, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Guatemala/Output/PS_data_piloto_v1/D_pan_recs_data_2026-03-31.csv'],
 'partitions_values': {}}

In [239]:
final_rec["cod_compania"] = final_rec["cod_compania"].astype(int)
final_rec["cod_sucursal"] = final_rec["cod_sucursal"].astype(int)
final_rec["cod_cliente"] = final_rec["cod_cliente"].astype(int)
final_rec["cod_modulo"] = final_rec["cod_modulo"].astype(int)
final_rec["sku"] = final_rec["sku"].astype(int)

### 5.8.2 Para subir a PS

In [240]:
recomendaciones_para_salesforce = final_rec[
    ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "sku"]
]
recomendaciones_para_salesforce = recomendaciones_para_salesforce.drop_duplicates()
recomendaciones_para_salesforce["Pais"] = "GT"
recomendaciones_para_salesforce["Cajas"] = int(1)
recomendaciones_para_salesforce["Unidades"] = int(0)
recomendaciones_para_salesforce["Fecha"] = fecha_tomorrow
recomendaciones_para_salesforce = recomendaciones_para_salesforce[
    [
        "Pais",
        "cod_compania",
        "cod_sucursal",
        "cod_cliente",
        "cod_modulo",
        "sku",
        "Cajas",
        "Unidades",
        "Fecha",
    ]
]
recomendaciones_para_salesforce.columns = [
    "Pais",
    "Compania",
    "Sucursal",
    "Cliente",
    "Modulo",
    "Producto",
    "Cajas",
    "Unidades",
    "Fecha",
]

In [241]:
recomendaciones_para_salesforce["Compania"] = recomendaciones_para_salesforce[
    "Compania"
].apply(lambda x: str(int(x)).rjust(4, "0"))
recomendaciones_para_salesforce["Sucursal"] = recomendaciones_para_salesforce[
    "Sucursal"
].apply(lambda x: str(int(x)).rjust(2, "0"))

In [242]:
recomendaciones_para_salesforce.dtypes

Pais        object
Compania     int64
Sucursal     int64
Cliente      int64
Modulo       int64
Producto     int64
Cajas        int64
Unidades     int64
Fecha       object
dtype: object

In [243]:
rutas_cliente = df_ventas[["cod_cliente", "cod_ruta"]].drop_duplicates()

In [244]:
recomendaciones_para_salesforce.Sucursal.unique()

array([], dtype=int64)

In [245]:
recomendaciones_para_salesforce.Compania.unique()

array([], dtype=int64)

In [246]:
recomendaciones_para_salesforce.Cliente.nunique()

0

In [247]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha


In [248]:
recomendaciones_para_salesforce.Cliente.nunique()

0

In [249]:
PS_VAL_total_final_clientes_recomendados = (
    recomendaciones_para_salesforce.Cliente.nunique()
)
PS_VAL_total_final_clientes_recomendados

0

In [250]:
PS_VAL_promedio_final_recomendaciones_por_cliente = (
    recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()
)
PS_VAL_promedio_final_recomendaciones_por_cliente

nan

In [251]:
recomendaciones_para_salesforce.groupby("Sucursal")["Producto"].nunique()

Series([], Name: Producto, dtype: int64)

In [252]:
PS_VAL_total_final_skus_recomendados = (
    recomendaciones_para_salesforce.Producto.nunique()
)
PS_VAL_total_final_skus_recomendados

0

**Conteo de marcas recomendadas**

In [253]:
count = (
    final_rec.groupby("id_cliente")["marca_rec"]
    .apply(lambda x: "-".join((x)))
    .reset_index(name="marcas_ordenadas")
)
count = (
    count.groupby("marcas_ordenadas")["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
)

In [254]:
PS_VAL_total_combinacion_rec_por_marca = count.shape[0]
PS_VAL_total_combinacion_rec_por_marca

0

**Conteo de SKUs recomendadas**

In [255]:
count2 = (
    recomendaciones_para_salesforce.groupby("Cliente")["Producto"]
    .apply(lambda x: "-".join(x.astype(str)))
    .reset_index(name="marcas_ordenadas")
)
count2 = (
    count2.groupby("marcas_ordenadas")["Cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
)
count2["f"] = count2["Cliente"] / count2["Cliente"].sum()
count2["fa"] = count2["f"].cumsum()

In [256]:
PS_VAL_total_combinacion_rec_por_sku = count2.shape[0]
PS_VAL_total_combinacion_rec_por_sku

0

In [257]:
PS_VAL_total_combinacion_rec_por_sku_al_80 = count2[count2["fa"] <= 0.8].shape[0]
PS_VAL_total_combinacion_rec_por_sku_al_80

0

In [258]:
recomendaciones_para_salesforce["Cliente"] = recomendaciones_para_salesforce[
    "Cliente"
].astype(str)
rutas_cliente["cod_cliente"] = rutas_cliente["cod_cliente"].astype(str)

In [259]:
pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
)

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,cod_cliente,cod_ruta


In [260]:
PS_VAL_rutas_skus_clientes_por_sucursal = pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
)
PS_VAL_rutas_skus_clientes_por_sucursal = (
    PS_VAL_rutas_skus_clientes_por_sucursal.groupby("Sucursal")
    .agg({"cod_ruta": "nunique", "Cliente": "nunique", "Producto": "nunique"})
    .reset_index()
)
PS_VAL_rutas_skus_clientes_por_sucursal.columns = [
    "Sucursal",
    "cant_rutas",
    "cant_clientes",
    "cant_skus",
]
PS_VAL_rutas_skus_clientes_por_sucursal

,Sucursal,cant_rutas,cant_clientes,cant_skus


In [261]:
print(f"RESUMEN PS para {fecha_tomorrow}")
print(f"*" * 20, "Metricas Iniciales", "*" * 20)
print(f"{'PS_VAL_total_clientes_venta:':<70}{PS_VAL_total_clientes_venta}")
print(f"{'PS_VAL_total_clientes_visita:':<70}{PS_VAL_total_clientes_visita}")
print(f"{'PS_VAL_total_clientes_visita_trad:':<70}{PS_VAL_total_clientes_visita_trad}")
print(f"{'PS_VAL_ultima_venta:':<70}{PS_VAL_ultima_venta}")
print(f"{'PS_VAL_ultima_visita:':<70}{PS_VAL_ultima_visita}")
print(f"{'PS_VAL_total_skus_venta:':<70}{PS_VAL_total_skus_venta}")
print(f"*" * 20, "Metricas del Proceso", "*" * 20)
print(f"{'PS_VAL_total_clientes_merged:':<70}{PS_VAL_total_clientes_merged}")
print(f"{'PS_VAL_total_fechas_unicas:':<70}{PS_VAL_total_fechas_unicas}")
print(f"{'PS_VAL_total_skus_merged:':<70}{PS_VAL_total_skus_merged}")
print(
    f"{'PS_VAL_total_clientes_imp_netovta_positivo:':<70}{PS_VAL_total_clientes_imp_netovta_positivo}"
)
print(
    f"{'PS_VAL_total_clientes_para_manhana:':<70}{PS_VAL_total_clientes_para_manhana}"
)
print(
    f"{'PS_VAL_total_skus_venta_para_manhana:':<70}{PS_VAL_total_skus_venta_para_manhana}"
)
# print(f"{'PS_VAL_total_skus_recomendados:':<70}{PS_VAL_total_skus_recomendados}")
# print(
#     f"{'PS_VAL_total_clientes_recomendados:':<70}{PS_VAL_total_clientes_recomendados}"
# )
# print(
#     f"{'PS_VAL_total_skus_recomendados_con_stock:':<70}{PS_VAL_total_skus_recomendados_con_stock}"
# )
# print(
#     f"{'PS_VAL_total_skus_recomendados_stock_y_activos:':<70}{PS_VAL_total_skus_recomendados_stock_y_activos}"
# )
# print(
#     f"{'PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda:':<70}{PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda}"
# )
print(f"*" * 20, "Metricas Finales", "*" * 20)
print(
    f"{'PS_VAL_total_final_skus_recomendados:':<70}{PS_VAL_total_final_skus_recomendados}"
)
print(
    f"{'PS_VAL_total_final_clientes_recomendados:':<70}{PS_VAL_total_final_clientes_recomendados}"
)
print(
    f"{'PS_VAL_promedio_recs_por_cliente_modelo:':<70}{PS_VAL_promedio_recs_por_cliente_modelo}"
)
print(
    f"{'PS_VAL_promedio_recs_por_cliente_stock:':<70}{PS_VAL_promedio_recs_por_cliente_stock}"
)
print(
    f"{'PS_VAL_promedio_recs_por_cliente_negocio:':<70}{PS_VAL_promedio_recs_por_cliente_negocio}"
)
print(
    f"{'PS_VAL_promedio_final_recomendaciones_por_cliente:':<70}{PS_VAL_promedio_final_recomendaciones_por_cliente}"
)
print(
    f"{'PS_VAL_total_combinacion_rec_por_marca:':<70}{PS_VAL_total_combinacion_rec_por_marca}"
)
print(f"{'PS_VAL_marcas_recomendadas:':<70}{PS_VAL_marcas_recomendadas}")
print(
    f"{'PS_VAL_total_combinacion_rec_por_sku:':<70}{PS_VAL_total_combinacion_rec_por_sku}"
)
print(
    f"{'PS_VAL_total_combinacion_rec_por_sku_al_80:':<70}{PS_VAL_total_combinacion_rec_por_sku_al_80}"
)
print(
    "PS_VAL_rutas_skus_clientes_por_sucursal:\n",
    PS_VAL_rutas_skus_clientes_por_sucursal,
)

RESUMEN PS para 2026-03-31
******************** Metricas Iniciales ********************
PS_VAL_total_clientes_venta:                                          50659
PS_VAL_total_clientes_visita:                                         50858
PS_VAL_total_clientes_visita_trad:                                    50858
PS_VAL_ultima_venta:                                                  ['2026-03-28']
PS_VAL_ultima_visita:                                                 ['2026-03-28']
PS_VAL_total_skus_venta:                                              119
******************** Metricas del Proceso ********************
PS_VAL_total_clientes_merged:                                         50333
PS_VAL_total_fechas_unicas:                                           333
PS_VAL_total_skus_merged:                                             119
PS_VAL_total_clientes_imp_netovta_positivo:                           50333
PS_VAL_total_clientes_para_manhana:                                   11
PS_V

In [262]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()

nan

In [263]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().value_counts()

Series([], Name: count, dtype: int64)

In [264]:
recomendaciones_para_salesforce.to_csv(
    f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv", index=False
)

In [265]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Guatemala/Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(recomendaciones_para_salesforce, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Guatemala/Output/PS_piloto_v1/D_base_pedidos_2026-03-31.csv'],
 'partitions_values': {}}

In [266]:
recomendaciones_para_salesforce.shape

(0, 9)

In [267]:
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["cod_cliente"] = (
    pan_ventas["cod_cliente"].astype(str)
)

In [268]:
val = pd.merge(recomendaciones_para_salesforce,pan_prod,left_on="Producto",right_on="cod_articulo_magic",how="left")
val = pd.merge(val,pan_ventas[["cod_compania","cod_cliente","cod_ruta","desc_sucursal","fecha_proceso"]].drop_duplicates(),left_on=["Compania","Cliente"],right_on=["cod_compania","cod_cliente"],how="left")

In [269]:
val

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,cod_articulo_magic,desc_articulo,cod_compania,cod_cliente,cod_ruta,desc_sucursal,fecha_proceso


In [270]:
val.cod_cliente.nunique()

0

In [271]:
fecha_hoy_val = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=0)
).strftime("%Y-%m-%d")
fecha_hoy_val

'2026-03-30'

In [272]:
# Se usa guarda este DF porque contiene info util para comercial
# Se activó cuando se requerian hacer validaciones
# val.to_csv(f"Val_comercial/D_CR_validacion_recomendaciones_{fecha_hoy_val}.csv",index=False)

### 5.8.3 Enviar Correo con Metricas de PS

In [273]:
# Define la zona horaria de Lima, Perú
lima_tz = pytz.timezone("America/Lima")

# Obtén la fecha y hora actual en la zona horaria de Lima
fecha_hoy_lima = datetime.now(lima_tz)
fecha_hoy = fecha_hoy_lima.strftime("%Y-%m-%d")
fecha_hoy

'2026-03-30'

In [274]:
# Resta un día para obtener la fecha de ayer
fecha_ayer_lima = fecha_hoy_lima - timedelta(days=1)

# Formatea la fecha en el formato año-mes-día
fecha_ayer = fecha_ayer_lima.strftime("%Y-%m-%d")
fecha_ayer

'2026-03-29'

In [275]:
# Obtener el día de la semana (1 es lunes, 7 es domingo)
dia_semana_hoy = (fecha_hoy_lima - timedelta(days=0)).weekday() + 1
dia_semana_hoy

1

In [276]:
# Si es lunes (1), restar 2 días para comparar
if dia_semana_hoy == 1:  # Lunes
    fecha_comparacion = fecha_ayer_lima - timedelta(days=1)
else:
    fecha_comparacion = fecha_ayer_lima

# Comprobar la fecha
if fecha_comparacion.strftime("%Y-%m-%d") == PS_VAL_ultima_venta[0]:
    mensaje_venta = f"Venta: PS listo. El archivo de ventas se encuentra actualizado hasta {fecha_comparacion.strftime('%Y-%m-%d')}."
else:
    mensaje_venta = f"Venta: PS FALTA. Actualizado hasta {PS_VAL_ultima_venta[0]}"

In [277]:
mensaje_venta

'Venta: PS listo. El archivo de ventas se encuentra actualizado hasta 2026-03-28.'

In [278]:
# Si es lunes (1), restar 2 días para comparar
if dia_semana_hoy == 1:  # Lunes
    fecha_comparacion = fecha_ayer_lima - timedelta(days=0)
else:
    fecha_comparacion = fecha_ayer_lima - timedelta(days=-1)

# Comprobar la fecha
if fecha_comparacion.strftime("%Y-%m-%d") == PS_VAL_ultima_visita[0]:
    mensaje_visita = f"Visita: PS listo. El archivo de visitas se encuentra actualizado hasta {fecha_comparacion.strftime('%Y-%m-%d')}."
else:
    mensaje_visita = f"Visita: PS FALTA. Actualizado hasta {PS_VAL_ultima_visita[0]}"

In [279]:
mensaje_visita

'Visita: PS FALTA. Actualizado hasta 2026-03-28'

In [280]:
# Credenciales para correo
remitente = "david.porta@ajegroup.com"
password = "pxqn kmuw drlq fvmv"

# Destinatario
asunto = f"Estado y Métricas PS para {fecha_tomorrow}"
# Lista de destinatarios
destinatarios = [
    # "jhon.poma@ajegroup.com",
    "david.porta@ajegroup.com",
    "wilmer.rodriguez@ajegroup.com",
    "masaru.gonzales@ajegroup.com",
    "gherald.barzola@ajegroup.com",
    "jorge.delgado.pe@csoluciones.pe",
    "melissa.cotrina@ajegroup.com"
]
#destinatarios = ["david.porta@ajegroup.com"]

# Cuerpo de Correo
msg = MIMEMultipart()
msg["From"] = remitente
msg["To"] = ", ".join(destinatarios)
msg["Subject"] = asunto

In [281]:
# Convertir el DataFrame a HTML si PS_VAL_rutas_skus_clientes_por_sucursal es un DataFrame
tabla_rutas_skus_clientes = PS_VAL_rutas_skus_clientes_por_sucursal.to_html(index=False)

# quitados
# <tr><td>SKUs con stock y activos:</td><td>{PS_VAL_total_skus_recomendados_stock_y_activos}</td></tr>
# <tr><td>SKUs con stock, activos y precio según el lambda:</td><td>{PS_VAL_total_skus_recomendados_stock_activos_y_precio_lambda}</td></tr>
# <tr><td>SKUs con stock:</td><td>{PS_VAL_total_skus_recomendados_con_stock}</td></tr>
# <h4>Métricas luego de correr el Modelo</h4>
# <table>
#     <tr><td>SKUs recomendados:</td><td>{PS_VAL_total_skus_recomendados}</td></tr>
#     <tr><td>Clientes recomendados:</td><td>{PS_VAL_total_clientes_recomendados}</td></tr>      
    
# </table>
# Agregar mensaje al cuerpo del correo en formato HTML
cuerpo_correo = f"""
<html>
<head>
    <style>
        table {{
            width: 50%;
            border-collapse: collapse;
        }}
        th, td {{
            padding: 10px;
            border: 1px solid black;
            text-align: left;
        }}
        th {{
            background-color: #f2f2f2;
        }}
    </style>
</head>
<body>
    <h2>GUATEMALA</h2>
    <h3>Status de subida de PS Guatemala para {fecha_tomorrow}:</h3>
    <p>{mensaje_venta}</p>
    <p>{mensaje_visita}</p>
    
    <h3>Pendientes:</h3>
    <p style="font-style: italic; color: #555;">Se pide apoyo a Masaru del Equipo Comercial para validar 
    que la subida de PS a Salesforce se realizó de manera satisfactoria.</p>
    
    <h3>Resumen PS Costa Rica para {fecha_tomorrow}:</h3>

    <h4>Métricas de Input</h4>
    <table>
        <tr><td>Clientes en Historico de Ventas:</td><td>{PS_VAL_total_clientes_venta}</td></tr>
        <tr><td>Clientes en archivo de Visitas:</td><td>{PS_VAL_total_clientes_visita}</td></tr>
        <tr><td>Clientes en archivo de Visitas y Canal Tradicional:</td><td>{PS_VAL_total_clientes_visita_trad}</td></tr>
        <tr><td>Fecha ultima venta liquidada:</td><td>{PS_VAL_ultima_venta}</td></tr>
        <tr><td>Fecha ultima visita:</td><td>{PS_VAL_ultima_visita}</td></tr>
        <tr><td>SKUs distintos en Historico de Ventas:</td><td>{PS_VAL_total_skus_venta}</td></tr>
    </table>

    <h4>Métricas del Proceso</h4>
    <table>
        <tr><td>Clientes combinados de ventas y visitas:</td><td>{PS_VAL_total_clientes_merged}</td></tr>
        <tr><td>Fechas de liquidacion unicas en ventas:</td><td>{PS_VAL_total_fechas_unicas}</td></tr>
        <tr><td>SKUs distintos en ventas y visitas:</td><td>{PS_VAL_total_skus_merged}</td></tr>
        <tr><td>Clientes con importe de venta positivo:</td><td>{PS_VAL_total_clientes_imp_netovta_positivo}</td></tr>
        <tr><td>Clientes para mañana:</td><td>{PS_VAL_total_clientes_para_manhana}</td></tr>
        <tr><td>SKUs para clientes de mañana:</td><td>{PS_VAL_total_skus_venta_para_manhana}</td></tr>
    </table>
    
    <h4>Métricas Finales</h4>
    <table>
        <tr><td>SKUs finales recomendados:</td><td>{PS_VAL_total_final_skus_recomendados}</td></tr>
        <tr><td>Clientes finales recomendados:</td><td>{PS_VAL_total_final_clientes_recomendados}</td></tr>
        <tr><td>Clientes finales recomendados + clientes con visita y sin venta:</td><td>{PS_VAL_total_clientes_con_visita_no_venta}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego del modelo:</td><td>{PS_VAL_promedio_recs_por_cliente_modelo}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de stock:</td><td>{PS_VAL_promedio_recs_por_cliente_stock}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de reglas de negocio:</td><td>{PS_VAL_promedio_recs_por_cliente_negocio}</td></tr>
        <tr><td>Promedio final de SKUs por cliente luego de reducir skus por segmento:</td><td>{PS_VAL_promedio_final_recomendaciones_por_cliente}</td></tr>
        <tr><td>Combinaciones de marca en recomendaciones:</td><td>{PS_VAL_total_combinacion_rec_por_marca}</td></tr>
        <tr><td>Marcas recomendadas:</td><td>{PS_VAL_marcas_recomendadas}</td></tr>
        <tr><td>Combinaciones de SKUs en recomendaciones:</td><td>{PS_VAL_total_combinacion_rec_por_sku}</td></tr>
        <tr><td>Combinaciones de SKUs al 80% de clientes:</td><td>{PS_VAL_total_combinacion_rec_por_sku_al_80}</td></tr>
    </table>

    <h4>Cantidad de Rutas, SKUs y Clientes por Sucursal:</h4>
    {tabla_rutas_skus_clientes}

</body>
</html>
"""

msg.attach(MIMEText(cuerpo_correo, "html"))

In [282]:
# Login SMTP de Gmail
server = smtplib.SMTP("smtp.gmail.com", 587)
server.starttls()
server.login(remitente, password)

(235, b'2.7.0 Accepted')

In [283]:
# Enviar correo
try:
    server.sendmail(remitente, destinatarios, msg.as_string())
    server.quit()
    print("Correo enviado")
except smtplib.SMTPException as e:
    print(f"Error SMTP al enviar correo: {e}")
except Exception as e:
    print(f"Error general al enviar correo: {e}")
except:
    print("Error al enviar correo")

Correo enviado


In [284]:
# # Detener el monitoreo
# running = False
# monitor_thread.join()

# # Graficar consumo de RAM y CPU
# fig, ax1 = plt.subplots(figsize=(10, 5))

# ax1.set_xlabel("Tiempo (segundos)")
# ax1.set_ylabel("RAM usada (GB)", color="tab:blue")
# ax1.plot(timestamps, mem_usage_gb, marker="o", linestyle="-", color="tab:blue", label="RAM (GB)")
# ax1.tick_params(axis="y", labelcolor="tab:blue")

# ax2 = ax1.twinx()  # Crear segundo eje Y para CPU
# ax2.set_ylabel("CPU usada (%)", color="tab:red")
# ax2.plot(timestamps, cpu_usage_percent, marker="x", linestyle="--", color="tab:red", label="CPU (%)")
# ax2.tick_params(axis="y", labelcolor="tab:red")

# fig.suptitle("Consumo de CPU y RAM del Notebook")
# ax1.grid()
# fig.tight_layout()
# plt.show()

In [285]:
# Apaga el Kernel para luego poder usar los demás notebooks sin llenar la memoria
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(False)  # True indica apagar completamente el kernel

{'status': 'ok', 'restart': False}